# 🏷️ Verificación de Tag `aws-apn-id` — CloudFormation Stack

Este notebook audita todos los recursos de un stack CloudFormation (incluyendo nested stacks)
y produce un reporte de:
- ✅ Recursos **con** el tag `aws-apn-id`
- ❌ Recursos **sin** el tag (pendientes)
- 📊 Resumen por tipo de recurso
- 📋 Lista detallada de ARNs pendientes de taguear

> **Solo lectura** — esta notebook no modifica nada.
> Para aplicar el tag usá `tag_apn_resources.py`.

## ⚙️ Configuración

In [ ]:
# ── Parámetros — modificá estos valores según el entorno ─────────────────────
STACK_NAME  = "agent-copilot-dev"          # Nombre del stack principal
REGION      = "us-east-1"                  # Región AWS
PROFILE     = None                         # AWS profile (None = credenciales del entorno)

TAG_KEY     = "aws-apn-id"
TAG_VALUE   = "pc:4da0ebcd14f35cd0p3n9zpm5l"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import subprocess, sys

# Instala dependencias faltantes sin interrumpir el flujo
for pkg in ("boto3", "pandas", "jinja2"):
    try:
        __import__(pkg if pkg != "jinja2" else "jinja2")
    except ImportError:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import boto3
import pandas as pd
from collections import defaultdict
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)

# Detecta si jinja2 está disponible para usar .style
try:
    import jinja2
    HAS_JINJA2 = True
except ImportError:
    HAS_JINJA2 = False

def show_df(df, caption="", highlight_col=None):
    """Muestra un DataFrame con estilos si jinja2 está disponible, si no en texto plano."""
    if caption:
        print(caption)
    if HAS_JINJA2 and highlight_col and highlight_col in df.columns:
        def _hl(row):
            color = "#fff3cd" if row[highlight_col] > 0 else "#d4edda"
            return [f"background-color: {color}"] * len(row)
        display(df.style.apply(_hl, axis=1).set_properties(**{"text-align": "left"}))
    else:
        display(df)

session_kwargs = {"region_name": REGION}
if PROFILE:
    session_kwargs["profile_name"] = PROFILE

session = boto3.Session(**session_kwargs)
cfn     = session.client("cloudformation")
tagging = session.client("resourcegroupstaggingapi")
bedrock = session.client("bedrock")
account = session.client("sts").get_caller_identity()["Account"]

print(f"✅ Sesión AWS iniciada — cuenta: {account}  región: {REGION}")
print(f"   Stack a auditar : {STACK_NAME}")
print(f"   Tag esperado    : {TAG_KEY} = {TAG_VALUE}")
print(f"   Estilos con color: {'✅ jinja2 disponible' if HAS_JINJA2 else '⚠️  sin jinja2 — display en texto plano'}")

## 🌐 Coverage snapshot — recursos con el tag en la cuenta

Sanity check rápido: cuántos recursos ya tienen `aws-apn-id` en toda la región, agrupados
por servicio. Útil para confirmar que el tag llegó a la cuenta antes de hacer el audit completo.

In [ ]:
def get_all_tagged_resources(tagging_client, tag_key, tag_value):
    """Devuelve todos los recursos de la cuenta/región que tienen tag_key=tag_value."""
    resources, token = [], None
    while True:
        kwargs = {"TagFilters": [{"Key": tag_key, "Values": [tag_value]}]}
        if token:
            kwargs["PaginationToken"] = token
        resp  = tagging_client.get_resources(**kwargs)
        resources.extend(resp["ResourceTagMappingList"])
        token = resp.get("PaginationToken")
        if not token:
            break
    return resources

tagged_global      = get_all_tagged_resources(tagging, TAG_KEY, TAG_VALUE)
tagged_arns_global = {r["ResourceARN"] for r in tagged_global}

print(f"Total recursos con {TAG_KEY}={TAG_VALUE}: {len(tagged_global)}\n")

by_service = defaultdict(int)
for r in tagged_global:
    parts   = r["ResourceARN"].split(":")
    service = parts[2] if len(parts) > 2 else "unknown"
    by_service[service] += 1

for service in sorted(by_service, key=lambda s: -by_service[s]):
    print(f"  {service:<30} {by_service[service]:>4} recursos")

## 📦 Paso 1 — Descubrir nested stacks

In [ ]:
def get_all_stacks(cfn_client, stack_id: str, collected: dict = None, depth: int = 0) -> dict:
    """
    Recorre el árbol de nested stacks recursivamente.
    Retorna dict { short_name -> stack_id (ARN o nombre) }
    """
    if collected is None:
        collected = {}

    short_name = stack_id.split(":stack/")[1].split("/")[0] if stack_id.startswith("arn:") else stack_id

    if short_name in collected:
        return collected

    collected[short_name] = stack_id
    indent = "  " * depth
    print(f"{indent}📦 {short_name}")

    try:
        paginator = cfn_client.get_paginator("list_stack_resources")
        for page in paginator.paginate(StackName=stack_id):
            for resource in page["StackResourceSummaries"]:
                if resource["ResourceType"] == "AWS::CloudFormation::Stack":
                    status    = resource.get("ResourceStatus", "")
                    nested_id = resource.get("PhysicalResourceId", "")
                    if nested_id and "DELETE" not in status:
                        get_all_stacks(cfn_client, nested_id, collected, depth + 1)
    except Exception as e:
        print(f"{indent}  ⚠️  No se pudo listar '{short_name}': {e}")

    return collected


print(f"Escaneando árbol de stacks desde '{STACK_NAME}'...\n")
all_stacks = get_all_stacks(cfn, STACK_NAME)
print(f"\n→ {len(all_stacks)} stacks encontrados en total")

## 🔍 Paso 2 — Recolectar recursos y verificar tags

In [ ]:
def get_resources_for_stack(tagging_client, stack_name: str) -> list:
    resources = []
    try:
        paginator = tagging_client.get_paginator("get_resources")
        for page in paginator.paginate(
            TagFilters=[{"Key": "aws:cloudformation:stack-name", "Values": [stack_name]}],
            ResourcesPerPage=100,
        ):
            for r in page["ResourceTagMappingList"]:
                arn   = r["ResourceARN"]
                tags  = {t["Key"]: t["Value"] for t in r.get("Tags", [])}
                parts = arn.split(":")
                resource_type = f"{parts[2]}:{parts[5]}" if len(parts) > 5 and parts[5] else parts[2]
                apn_value = tags.get(TAG_KEY)
                resources.append({
                    "stack"         : stack_name,
                    "arn"           : arn,
                    "resource_type" : resource_type,
                    "has_apn_tag"   : apn_value is not None,
                    "apn_tag_value" : apn_value or "",
                    "tag_ok"        : apn_value == TAG_VALUE,
                })
    except Exception as e:
        print(f"  ⚠️  Error en '{stack_name}': {e}")
    return resources


all_resources = []
seen_arns     = set()

print("Recolectando recursos...\n")
for stack_name in sorted(all_stacks.keys()):
    resources = get_resources_for_stack(tagging, stack_name)
    new       = [r for r in resources if r["arn"] not in seen_arns]
    seen_arns.update(r["arn"] for r in new)
    all_resources.extend(new)
    tagged   = sum(1 for r in new if r["tag_ok"])
    untagged = len(new) - tagged
    print(f"  {stack_name:<60} {len(new):>4} recursos  |  ✅ {tagged:>3}  ❌ {untagged:>3}")

df = pd.DataFrame(all_resources)
print(f"\nTotal recursos únicos encontrados (vía CFN): {len(df)}")

## 🤖 Paso 3 — Bedrock Inference Profiles (chequeo directo)

Los Inference Profiles **no aparecen** en la Tagging API cuando se filtra por
`aws:cloudformation:stack-name`. Se verifican directamente con `bedrock.list_tags_for_resource()`.

In [ ]:
bedrock_results = []

try:
    inf_profiles, token = [], None
    while True:
        kwargs = {"typeEquals": "APPLICATION"}
        if token:
            kwargs["nextToken"] = token
        resp  = bedrock.list_inference_profiles(**kwargs)
        inf_profiles.extend(resp.get("inferenceProfileSummaries", []))
        token = resp.get("nextToken")
        if not token:
            break

    for p in inf_profiles:
        arn  = p["inferenceProfileArn"]
        name = p["inferenceProfileName"]
        try:
            tags = {t["key"]: t["value"]
                    for t in bedrock.list_tags_for_resource(resourceARN=arn).get("tags", [])}
        except Exception:
            tags = {}
        apn_value = tags.get(TAG_KEY)
        bedrock_results.append({
            "nombre"        : name,
            "arn"           : arn,
            "tag_ok"        : apn_value == TAG_VALUE,
            "apn_tag_value" : apn_value or "(sin tag)",
        })

    ok  = sum(1 for r in bedrock_results if r["tag_ok"])
    bad = len(bedrock_results) - ok
    print(f"[Bedrock Inference Profiles]  total: {len(bedrock_results)}  ✅ {ok}  ❌ {bad}")
    for r in bedrock_results:
        icon = "✅" if r["tag_ok"] else "❌"
        print(f"  {icon}  {r['nombre']}")
        if not r["tag_ok"]:
            print(f"      {r['arn']}")
            print(f"      tag actual: {r['apn_tag_value']}")

except Exception as e:
    print(f"⚠️  No se pudo listar Inference Profiles: {e}")

## 📊 Paso 4 — Resumen ejecutivo

In [ ]:
if df.empty:
    print("⚠️  No se encontraron recursos CFN. Verificá permisos y nombre del stack.")
else:
    total     = len(df)
    tagged_ok = int(df["tag_ok"].sum())
    has_tag   = int(df["has_apn_tag"].sum())
    wrong_val = has_tag - tagged_ok
    missing   = total - has_tag
    pct       = tagged_ok / total * 100

    bk_total = len(bedrock_results)
    bk_ok    = sum(1 for r in bedrock_results if r["tag_ok"])
    bk_bad   = bk_total - bk_ok

    grand_total = total + bk_total
    grand_ok    = tagged_ok + bk_ok
    grand_pct   = grand_ok / grand_total * 100 if grand_total else 0

    print("═" * 60)
    print(f"  RESUMEN — {STACK_NAME}")
    print("═" * 60)
    print(f"  Recursos CFN (vía Tagging API)")
    print(f"    Total auditados              : {total:>5}")
    print(f"    ✅ Con tag correcto           : {tagged_ok:>5}   ({pct:.1f}%)")
    print(f"    ⚠️  Tag con valor incorrecto  : {wrong_val:>5}")
    print(f"    ❌ Sin tag                   : {missing:>5}")
    print(f"  Bedrock Inference Profiles (directo)")
    print(f"    Total                        : {bk_total:>5}")
    print(f"    ✅ Con tag correcto           : {bk_ok:>5}")
    print(f"    ❌ Sin tag / valor incorrecto : {bk_bad:>5}")
    print("─" * 60)
    print(f"  TOTAL GLOBAL  {grand_ok}/{grand_total}  ({grand_pct:.1f}% cobertura)")
    print("═" * 60)

    if grand_ok == grand_total:
        print("\n🎉 ¡Todos los recursos están correctamente tagueados!")
    else:
        pendientes = grand_total - grand_ok
        print(f"\n⚠️  {pendientes} recurso(s) pendientes de taguear.")
        print(f"   → CFN: ejecutá tag_apn_resources.py")
        if bk_bad:
            print(f"   → Bedrock Inference Profiles: taguear manualmente (ver Paso 3)")

## 📋 Paso 5 — Desglose por tipo de recurso

In [ ]:
if not df.empty:
    summary = (
        df.groupby("resource_type")
        .agg(
            total   = ("arn",    "count"),
            con_tag = ("tag_ok", "sum"),
        )
        .assign(sin_tag   = lambda x: x["total"] - x["con_tag"])
        .assign(cobertura = lambda x: (x["con_tag"] / x["total"] * 100).round(1).astype(str) + "%")
        .sort_values(["sin_tag", "total"], ascending=[False, False])
        .reset_index()
        .rename(columns={
            "resource_type": "Tipo de recurso",
            "total"        : "Total",
            "con_tag"      : "Con tag ✅",
            "sin_tag"      : "Sin tag ❌",
            "cobertura"    : "Cobertura",
        })
    )
    show_df(summary, caption=f"Cobertura de '{TAG_KEY}' por tipo de recurso:", highlight_col="Sin tag ❌")

## ❌ Paso 6 — Recursos sin tag (detalle)

In [ ]:
if not df.empty:
    missing_df = df[~df["tag_ok"]][["stack", "resource_type", "arn", "apn_tag_value"]].copy()
    missing_df.columns = ["Stack", "Tipo", "ARN", "Valor actual del tag"]
    missing_df["Valor actual del tag"] = missing_df["Valor actual del tag"].replace("", "(sin tag)")

    if missing_df.empty:
        print("✅ No hay recursos CFN pendientes de taguear.")
    else:
        print(f"❌ {len(missing_df)} recursos CFN sin el tag correcto:\n")
        show_df(missing_df.reset_index(drop=True), caption="Recursos que requieren el tag aws-apn-id:")

## 📤 Paso 7 — Exportar lista de pendientes (opcional)

In [ ]:
if not df.empty:
    missing_export = df[~df["tag_ok"]].copy()

    bk_missing = [r for r in bedrock_results if not r["tag_ok"]]
    if bk_missing:
        bk_df = pd.DataFrame([{
            "stack"        : "bedrock-inference-profile",
            "arn"          : r["arn"],
            "resource_type": "bedrock:inference-profile",
            "has_apn_tag"  : r["apn_tag_value"] != "(sin tag)",
            "apn_tag_value": r["apn_tag_value"],
            "tag_ok"       : False,
        } for r in bk_missing])
        missing_export = pd.concat([missing_export, bk_df], ignore_index=True)

    if not missing_export.empty:
        output_path = f"apn_tagging_pendientes_{STACK_NAME}.csv"
        missing_export.to_csv(output_path, index=False)
        print(f"✅ Exportado: {output_path}  ({len(missing_export)} recursos)")
        print(f"\nARNs pendientes:")
        for arn in missing_export["arn"].tolist():
            print(f"  {arn}")
    else:
        print("✅ No hay recursos pendientes — no se genera archivo.")